In [ ]:
import os,glob,gc,time,psutil
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split,StratifiedKFold

gpus=tf.config.list_physical_devices("GPU")
for gpu in gpus: tf.config.experimental.set_memory_growth(gpu,True)
print("GPU:",gpus)

SEED=123
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_DIR=r"D:/GCN/Brain_Tumor/four_class"
IMG_SIZE=(150,150)
DOMAIN_SIZE=(64,64)
BATCH_SIZE=32
K_FOLDS=5
LR_CNN=1e-3
EPOCHS_CNN= 50
AUGMENT=True

class_names=sorted([d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR,d))])
class_to_idx={c:i for i,c in enumerate(class_names)}

paths=[]
labels=[]

for c in class_names:
    for f in glob.glob(os.path.join(DATA_DIR,c,"*")):
        if f.lower().endswith((".jpg",".jpeg",".png",".bmp",".tif",".tiff")):
            paths.append(f)
            labels.append(class_to_idx[c])

paths=np.array(paths)
labels=np.array(labels,dtype=np.int32)

N=len(paths)
NUM_CLASSES=len(class_names)

print("Total images:",N)
print("Classes:",class_names)
print("Class distribution:",np.unique(labels,return_counts=True))

all_indices=np.arange(N)

dev_indices,test_indices=train_test_split(all_indices,test_size=0.1,stratify=labels,random_state=SEED)
print("Development:",len(dev_indices));print("Test:",len(test_indices))
skf=StratifiedKFold(n_splits=K_FOLDS,shuffle=True,random_state=SEED)

def load_img(path,label):
    img=tf.io.read_file(path)
    img=tf.image.decode_image(img,channels=3,expand_animations=False)
    img=tf.image.convert_image_dtype(img,tf.float32)
    img=tf.image.resize(img,IMG_SIZE)
    img.set_shape([IMG_SIZE[0],IMG_SIZE[1],3])
    return img,label

def augment_img(img,label):
    if AUGMENT:
        img=tf.image.random_flip_left_right(img)
        img=tf.image.rot90(img,tf.random.uniform([],0,4,dtype=tf.int32))
    return img,label

def make_dataset(indices,training=False,shuffle=False):
    ds=tf.data.Dataset.from_tensor_slices((paths[indices],labels[indices]))
    if shuffle:
        ds=ds.shuffle(len(indices),seed=SEED)
    ds=ds.map(load_img,num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds=ds.map(augment_img,num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

def build_gcnn4():
    inp=tf.keras.Input(shape=(*IMG_SIZE,3))
    x=tf.keras.layers.Conv2D(32,3,padding="same",activation="relu")(inp)
    x=tf.keras.layers.MaxPooling2D()(x)
    x=tf.keras.layers.Conv2D(64,3,padding="same",activation="relu")(x)
    x=tf.keras.layers.MaxPooling2D()(x)
    x=tf.keras.layers.Conv2D(128,3,padding="same",activation="relu")(x)
    x=tf.keras.layers.MaxPooling2D()(x)
    x=tf.keras.layers.Conv2D(256,3,padding="same",activation="relu")(x)
    x=tf.keras.layers.MaxPooling2D()(x)
    x=tf.keras.layers.GlobalAveragePooling2D()(x)
    feature=tf.keras.layers.Dense(512,activation="relu",name="deep_feature")(x)
    output=tf.keras.layers.Dense(NUM_CLASSES,activation="softmax")(feature)
    model=tf.keras.Model(inp,output,name="gCNN4")
    backbone=tf.keras.Model(inp,feature,name="gCNN4_backbone")
    return model,backbone

def extract_deep_features(backbone,dataset):
    X=[]
    Y=[]
    for xb,yb in dataset:
        X.append(backbone(xb,training=False).numpy())
        Y.append(yb.numpy())
    return np.vstack(X).astype(np.float32),np.concatenate(Y)

def entropy_feature(gray):
    hist=tf.histogram_fixed_width(gray,[0.0,1.0],nbins=32)
    hist=tf.cast(hist,tf.float32)
    prob=hist/tf.reduce_sum(hist)
    prob=tf.where(prob>0,prob,tf.ones_like(prob))
    return tf.cast(-tf.reduce_sum(prob*tf.math.log(prob)),tf.float32)

def extract_raw_domain(path):
    img=tf.io.read_file(path)
    img=tf.image.decode_image(img,channels=3,expand_animations=False)
    img=tf.image.convert_image_dtype(img,tf.float32)
    img=tf.image.resize(img,IMG_SIZE)
    gray=tf.image.rgb_to_grayscale(img)
    raw=tf.reshape(gray,[-1])
    small=tf.image.resize(gray,DOMAIN_SIZE)
    mean=tf.reduce_mean(small)
    std=tf.math.reduce_std(small)
    variance=tf.math.reduce_variance(small)
    minimum=tf.reduce_min(small)
    maximum=tf.reduce_max(small)
    sobel=tf.image.sobel_edges(tf.expand_dims(small,0))
    gradient=tf.sqrt(tf.reduce_sum(tf.square(sobel),axis=-1))
    edge=tf.reduce_mean(gradient)
    entropy=entropy_feature(small)
    domain=tf.stack([mean,std,variance,minimum,maximum,edge,entropy])
    return raw,tf.cast(domain,tf.float32)

def extract_raw_domain_features():
    raw_features=[]
    domain_features=[]
    start=time.time()

    for i,p in enumerate(paths):
        raw,domain=extract_raw_domain(p)
        raw_features.append(raw.numpy())
        domain_features.append(domain.numpy())
        if i%500==0:
            print("Processed:",i)

    print("Feature extraction time:",(time.time()-start)/60,"minutes")
    return np.asarray(raw_features,dtype=np.float32),np.asarray(domain_features,dtype=np.float32)

print("PART 1 COMPLETED")

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import coo_matrix,csr_matrix
from scipy.sparse.csgraph import connected_components
from tensorflow.keras import layers,Input,Model,regularizers
from spektral.layers import GCNConv
import numpy as np
import tensorflow as tf
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import coo_matrix,csr_matrix
from scipy.sparse.csgraph import connected_components
from tensorflow.keras import layers,Input,Model,regularizers
from spektral.layers import GCNConv

SEED=123
K_NEIGHBORS=12
DROPOUT=0.3
LR_GCN=0.005
NUM_CLASSES=4

def fold_feature_fusion(X_deep,X_raw,X_domain,train_idx,val_idx,test_idx):
    def pca_branch(X):
        n=min(128,X.shape[1],len(train_idx)-1)
        pca=PCA(n_components=n,random_state=SEED)
        scaler=StandardScaler()
        tr=pca.fit_transform(X[train_idx])
        va=pca.transform(X[val_idx])
        te=pca.transform(X[test_idx])
        tr=scaler.fit_transform(tr)
        va=scaler.transform(va)
        te=scaler.transform(te)
        return tr,va,te
    deep_tr,deep_va,deep_te=pca_branch(X_deep)
    raw_tr,raw_va,raw_te=pca_branch(X_raw)
    scaler=StandardScaler()
    domain_tr=scaler.fit_transform(X_domain[train_idx])
    domain_va=scaler.transform(X_domain[val_idx])
    domain_te=scaler.transform(X_domain[test_idx])
    X_train=np.concatenate([deep_tr,raw_tr,domain_tr],axis=1)
    X_val=np.concatenate([deep_va,raw_va,domain_va],axis=1)
    X_test=np.concatenate([deep_te,raw_te,domain_te],axis=1)
    scaler=StandardScaler()
    X_train=scaler.fit_transform(X_train)
    X_val=scaler.transform(X_val)
    X_test=scaler.transform(X_test)
    return X_train.astype(np.float32),X_val.astype(np.float32),X_test.astype(np.float32)

def build_knn_cosine_graph(X,k=K_NEIGHBORS):
    n=len(X)
    k=min(k,n-1)
    knn=NearestNeighbors(n_neighbors=k+1,metric="cosine").fit(X)
    dist,idx=knn.kneighbors(X)
    rows=[];cols=[];vals=[]
    for i in range(n):
        for j,d in zip(idx[i],dist[i]):
            if i!=j:
                w=1-float(d)
                if w>0:
                    rows.append(i)
                    cols.append(j)
                    vals.append(w)
    A=coo_matrix((vals,(rows,cols)),shape=(n,n),dtype=np.float32).tocsr()
    A=(A+A.T).tocsr()
    A.sum_duplicates()
    A.setdiag(1)
    A.eliminate_zeros()
    return A

def build_transductive_knn_graph(X):
    return build_knn_cosine_graph(X,K_NEIGHBORS)

def build_inductive_knn_graph(X_train,X_new):
    n_train=len(X_train)
    A_train=build_knn_cosine_graph(X_train,K_NEIGHBORS).tocoo()
    rows=list(A_train.row)
    cols=list(A_train.col)
    vals=list(A_train.data)
    knn=NearestNeighbors(n_neighbors=min(K_NEIGHBORS,n_train),metric="cosine").fit(X_train)
    dist,idx=knn.kneighbors(X_new)
    for i in range(len(X_new)):
        node=n_train+i
        for j,d in zip(idx[i],dist[i]):
            w=1-float(d)
            if w>0:
                rows.extend([node,j])
                cols.extend([j,node])
                vals.extend([w,w])
    A=coo_matrix((vals,(rows,cols)),shape=(n_train+len(X_new),n_train+len(X_new)),dtype=np.float32).tocsr()
    A.sum_duplicates()
    A.setdiag(1)
    A.eliminate_zeros()
    return A

def graph_statistics(A):
    B=A.copy()
    B.data=np.ones_like(B.data)
    degree=np.asarray(B.sum(1)).ravel()
    edges=B.nnz//2
    density=edges/(B.shape[0]*(B.shape[0]-1)/2)
    comp,_=connected_components(B,directed=False)
    print(f"Nodes:{A.shape[0]} | Edges:{edges} | Degree:{degree.mean():.2f} | Density:{density:.5f} | Components:{comp}")

def build_gcn(feature_dim,num_nodes):
    X_in=Input(shape=(feature_dim,),name="features")
    A_in=Input(shape=(num_nodes,num_nodes),sparse=True,name="graph")
    x=GCNConv(64,activation="relu",kernel_regularizer=regularizers.l2(5e-4))([X_in,A_in])
    x=layers.Dropout(DROPOUT)(x)
    out=GCNConv(NUM_CLASSES,activation="softmax")([x,A_in])
    model=Model([X_in,A_in],out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(LR_GCN),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
        weighted_metrics=["accuracy"]
    )
    return model

print("PART 2 FINAL")

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical
from spektral.utils.convolution import gcn_filter
from sklearn.metrics import accuracy_score
from scipy.sparse import csr_matrix
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
EPOCHS_GCN=100
NUM_CLASSES=len(class_names)

fold_results={"transductive":[],"inductive":[]}
results={m:{"true":[],"pred":[],"prob":[],"fold_acc":[],"histories":[],"cnn_time":[],"gcn_time":[]} for m in ["transductive","inductive"]}
def sparse_adj(A):
    A=csr_matrix(A,dtype=np.float32)
    return tf.sparse.reorder(tf.sparse.SparseTensor(np.vstack(A.nonzero()).T,A.data,A.shape))

def graph_stats(A):
    if isinstance(A,tf.SparseTensor):
        idx,val,shape=A.indices.numpy(),A.values.numpy(),A.dense_shape.numpy()
        A=csr_matrix((val,(idx[:,0],idx[:,1])),shape=shape)

    B=A.copy();B.data=np.ones_like(B.data)
    nodes=A.shape[0];edges=(A.nnz-nodes)//2
    degree=np.asarray(B.sum(1)).ravel()-1
    components,cc=connected_components(A,directed=False)

    return {"nodes":nodes,"edges":edges,"avg_degree":degree.mean(),"min_degree":degree.min(),"max_degree":degree.max(),"density":edges/(nodes*(nodes-1)/2),"components":components,"largest_component":np.max(np.bincount(cc))}

    
def save_fold(mode,fold,yt,yp,prob,hist,A,ct,gt):
    acc=accuracy_score(yt,yp)
    fold_results[mode].append({"fold":fold,"true":yt,"pred":yp,"prob":prob,"history":hist.history,"graph":graph_stats(A),"cnn_time":ct,"gcn_time":gt,"accuracy":acc})
    results[mode]["true"].extend(yt);results[mode]["pred"].extend(yp);results[mode]["prob"].extend(prob)
    results[mode]["fold_acc"].append(acc);results[mode]["histories"].append(hist.history)
    results[mode]["cnn_time"].append(ct);results[mode]["gcn_time"].append(gt)

X_raw,X_domain=extract_raw_domain_features()

for fold,(tr,val) in enumerate(skf.split(dev_indices,labels[dev_indices]),1):
    
    print(f"\n========== FOLD {fold} ==========")
    train_idx,val_idx=dev_indices[tr],dev_indices[val]
    tf.keras.backend.clear_session();gc.collect()

    train_ds=make_dataset(train_idx,training=True,shuffle=True)
    val_ds=make_dataset(val_idx)
    cnn,backbone=build_gcnn4()

    cnn.compile(optimizer=tf.keras.optimizers.Adam(LR_CNN),loss="sparse_categorical_crossentropy",metrics=["accuracy"])

    t=time.time()
    cnn_hist=cnn.fit(train_ds,validation_data=val_ds,epochs=EPOCHS_CNN,callbacks=[EarlyStopping(monitor="val_loss",patience=10,restore_best_weights=True)],verbose=1)
    cnn_time=time.time()-t

    X_deep,_=extract_deep_features(backbone,make_dataset(np.arange(N)))

    del cnn,backbone,train_ds,val_ds
    gc.collect()

    X_train,X_val,X_test=fold_feature_fusion(X_deep,X_raw,X_domain,train_idx,val_idx,test_indices)

    # TRANSDUCTIVE
    print("Transductive")

    X_all=np.vstack([X_train,X_val,X_test])
    y_all=np.concatenate([labels[train_idx],labels[val_idx],labels[test_indices]])

    A_raw=csr_matrix(gcn_filter(build_transductive_knn_graph(X_all)),dtype=np.float32)
    A=sparse_adj(A_raw)

    Y=to_categorical(y_all,NUM_CLASSES)
    mask=np.zeros(len(y_all),dtype=np.float32)
    mask[:len(train_idx)]=1

    gcn=build_gcn(X_all.shape[1],len(X_all))

    t=time.time()
    hist=gcn.fit([X_all,A],Y,sample_weight=mask,epochs=EPOCHS_GCN,batch_size=len(X_all),shuffle=False,callbacks=[EarlyStopping(monitor="loss",patience=30,restore_best_weights=True)],verbose=1)
    gcn_time=time.time()-t

    prob=gcn([X_all,A],training=False).numpy()
    pred=np.argmax(prob,1)
    start=len(train_idx)+len(val_idx)

    save_fold("transductive",fold,y_all[start:],pred[start:],prob[start:],hist,A_raw,cnn_time,gcn_time)

    del gcn,A
    gc.collect()

    # INDUCTIVE
    print("Inductive")

    X_query=np.vstack([X_val,X_test])
    X_full=np.vstack([X_train,X_query])

    A_raw=csr_matrix(gcn_filter(build_inductive_knn_graph(X_train,X_query)),dtype=np.float32)
    A=sparse_adj(A_raw)

    Y_full=np.vstack([to_categorical(labels[train_idx],NUM_CLASSES),np.zeros((len(X_query),NUM_CLASSES))])
    mask=np.concatenate([np.ones(len(X_train)),np.zeros(len(X_query))])

    gcn=build_gcn(X_full.shape[1],len(X_full))

    t=time.time()
    hist=gcn.fit([X_full,A],Y_full,sample_weight=mask,epochs=EPOCHS_GCN,batch_size=len(X_full),shuffle=False,callbacks=[EarlyStopping(monitor="loss",patience=30,restore_best_weights=True)],verbose=1)
    gcn_time=time.time()-t

    prob=gcn([X_full,A],training=False).numpy()
    pred=np.argmax(prob,1)
    start=len(X_train)+len(X_val)

    save_fold("inductive",fold,labels[test_indices],pred[start:],prob[start:],hist,A_raw,cnn_time,gcn_time)

    final_model=gcn
    final_A=A
    final_X=X_full

del gcn,A
gc.collect()

print("TRAINING COMPLETED")
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd,numpy as np
from sklearn.metrics import *
from sklearn.preprocessing import label_binarize
from scipy.sparse.csgraph import connected_components
import time,psutil,os

def plot_cm(cm,title,norm=False):
    if norm: cm=cm.astype(float)/np.maximum(cm.sum(1,keepdims=True),1)
    plt.figure(figsize=(6,5));sns.heatmap(cm,annot=True,fmt=".2f" if norm else "d",cmap="Blues",xticklabels=class_names,yticklabels=class_names)
    plt.title(title);plt.xlabel("Predicted");plt.ylabel("True");plt.show()

def plot_roc(y_true,prob,title):
    y_bin=label_binarize(y_true,classes=np.arange(NUM_CLASSES));plt.figure(figsize=(6,5));aucs=[]
    for i in range(NUM_CLASSES):
        fpr,tpr,_=roc_curve(y_bin[:,i],prob[:,i]);a=auc(fpr,tpr);aucs.append(a);plt.plot(fpr,tpr,label=f"{class_names[i]}:{a:.3f}")
    fpr,tpr,_=roc_curve(y_bin.ravel(),prob.ravel());micro=auc(fpr,tpr)
    plt.plot(fpr,tpr,"--",label=f"Micro:{micro:.3f}");plt.plot([0,1],[0,1],"k--")
    plt.title(title+f" Macro:{np.mean(aucs):.3f}");plt.legend();plt.show()

def plot_history(history,title):
    for a,b in [("accuracy","val_accuracy"),("loss","val_loss")]:
        if a in history:
            plt.figure(figsize=(6,4));plt.plot(history[a])
            if b in history: plt.plot(history[b]);plt.legend(["Train","Validation"])
            plt.title(title+" "+a);plt.xlabel("Epoch");plt.ylabel(a);plt.show()

def evaluate_block(y_true,y_pred,prob,title):
    print("\n====================\n"+title+"\n====================")
    print(pd.DataFrame(classification_report(y_true,y_pred,target_names=class_names,output_dict=True,zero_division=0)).T)
    cm=confusion_matrix(y_true,y_pred)
    print("\nConfusion Matrix\n",cm)
    plot_cm(cm,title+" CM");plot_cm(cm,title+" Normalized CM",True);plot_roc(y_true,prob,title+" ROC")


# EACH FOLD RESULTS
for mode in ["transductive","inductive"]:
    print("\n\nMODEL:",mode.upper())
    for item in fold_results[mode]:
        fold=item["fold"]
        evaluate_block(item["true"],item["pred"],item["prob"],f"{mode} Fold {fold}")
        plot_history(item["history"],f"{mode} Fold {fold}")
        print("Graph statistics\n",item["graph"])


# COMBINED CV RESULTS
for mode in ["transductive","inductive"]:
    y_true=np.concatenate([x["true"] for x in fold_results[mode]])
    y_pred=np.concatenate([x["pred"] for x in fold_results[mode]])
    prob=np.vstack([x["prob"] for x in fold_results[mode]])

    evaluate_block(y_true,y_pred,prob,mode+" Combined CV")

    acc=[];loss=[]
    for x in fold_results[mode]:
        acc.extend(x["history"].get("accuracy",[]))
        loss.extend(x["history"].get("loss",[]))

    plot_history({"accuracy":acc,"loss":loss},mode+" Combined Training")


# FINAL TEST DATA
for mode in ["transductive","inductive"]:
    test_true=np.concatenate([x["true"] for x in fold_results[mode]])
    test_pred=np.concatenate([x["pred"] for x in fold_results[mode]])
    test_prob=np.vstack([x["prob"] for x in fold_results[mode]])
    evaluate_block(test_true,test_pred,test_prob,mode+" TEST DATA")

final_model=final_model
X_final=final_X
A_final=final_A
C=NUM_CLASSES

params=final_model.count_params()
model_size=params*4/(1024**2)

graph_info=graph_stats(A_final)
edges=graph_info["edges"]

gcn_flops=2*edges*X_final.shape[1]*64+2*edges*64*C
gcn_gflops=gcn_flops/1e9

start=time.time()
_=final_model([X_final,A_final],training=False)
inference_time=max(time.time()-start,1e-9)

idx_test=len(test_indices)
latency=(inference_time/idx_test)*1000
throughput=idx_test/inference_time

computational=pd.DataFrame([{
    "Model":"Hybrid-GCN-kNN",
    "Parameters":params,
    "Model_Size_MB":model_size,
    "Nodes":graph_info["nodes"],
    "Edges":graph_info["edges"],
    "Average_Degree":graph_info["avg_degree"],
    "Density":graph_info["density"],
    "Components":graph_info["components"],
    "Largest_Component":graph_info["largest_component"],
    "GCN_FLOPs":gcn_flops,
    "GCN_GFLOPs":gcn_gflops,
    "Inference_Time_sec":inference_time,
    "Latency_ms":latency,
    "Throughput_img_sec":throughput,
    "RAM_GB":psutil.Process(os.getpid()).memory_info().rss/1024**3
}])

print(computational)
computational.to_csv("Hybrid_GCN_kNN_Computational_Analysis.csv",index=False)

print("PART B COMPLETED")

In [ ]:
# ============================================================
# Hybrid-GCN-kNN
# PART 4: COMPLETE EVALUATION
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import *
from sklearn.preprocessing import label_binarize

def plot_cm(cm,title,norm=False):
    if norm: cm=cm.astype(float)/np.maximum(cm.sum(1,keepdims=True),1)
    plt.figure(figsize=(6,5))
    sns.heatmap(cm,annot=True,fmt=".2f" if norm else "d",cmap="Blues",xticklabels=class_names,yticklabels=class_names)
    plt.title(title);plt.xlabel("Predicted");plt.ylabel("True");plt.show()


def plot_roc(y_true,prob,title):
    y_bin=label_binarize(y_true,classes=np.arange(NUM_CLASSES))
    plt.figure(figsize=(6,5));aucs=[]
    for i in range(NUM_CLASSES):
        if len(np.unique(y_bin[:,i]))>1:
            fpr,tpr,_=roc_curve(y_bin[:,i],prob[:,i])
            a=auc(fpr,tpr);aucs.append(a)
            plt.plot(fpr,tpr,label=f"{class_names[i]} AUC={a:.3f}")
    fpr,tpr,_=roc_curve(y_bin.ravel(),prob.ravel())
    micro=auc(fpr,tpr);macro=np.mean(aucs)
    plt.plot(fpr,tpr,"--",label=f"Micro={micro:.3f}")
    plt.plot([0,1],[0,1],"k--")
    plt.title(title+f"\nMacro AUC={macro:.3f}")
    plt.legend();plt.show()


def plot_history(hist,title):
    if "accuracy" in hist:
        plt.figure(figsize=(6,4));plt.plot(hist["accuracy"])
        if "val_accuracy" in hist: plt.plot(hist["val_accuracy"])
        plt.title(title+" Accuracy");plt.xlabel("Epoch");plt.ylabel("Accuracy");plt.legend(["Train","Validation"]);plt.show()
    if "loss" in hist:
        plt.figure(figsize=(6,4));plt.plot(hist["loss"])
        if "val_loss" in hist: plt.plot(hist["val_loss"])
        plt.title(title+" Loss");plt.xlabel("Epoch");plt.ylabel("Loss");plt.legend(["Train","Validation"]);plt.show()


def evaluate_block(y_true,y_pred,prob,title):
    print("\n==============================\n"+title+"\n==============================")
    report=pd.DataFrame(classification_report(y_true,y_pred,target_names=class_names,output_dict=True,zero_division=0)).T
    print(report)
    cm=confusion_matrix(y_true,y_pred)
    print(cm)
    plot_cm(cm,title+" CM")
    plot_cm(cm,title+" Normalized CM",True)
    plot_roc(y_true,prob,title+" ROC")
    return report,cm


# EACH FOLD RESULTS
for mode in ["transductive","inductive"]:
    print("\n\n########",mode.upper(),"FOLD RESULTS ########")
    for i,h in enumerate(results[mode]["histories"]):
        print("\nFOLD",i+1)
        plot_history(h,f"{mode} Fold {i+1}")


# COMBINED ALL FOLDS
combined_reports={}

for mode in ["transductive","inductive"]:

    yt=np.array(results[mode]["true"])
    yp=np.array(results[mode]["pred"])
    prob=np.array(results[mode]["prob"])

    report,cm=evaluate_block(yt,yp,prob,mode+" Combined")
    combined_reports[mode]=report

    acc=[];loss=[]
    for h in results[mode]["histories"]:
        acc.extend(h.get("accuracy",[]))
        loss.extend(h.get("loss",[]))

    plt.figure(figsize=(6,4));plt.plot(acc);plt.title(mode+" Combined Accuracy Curve");plt.xlabel("Epoch");plt.ylabel("Accuracy");plt.show()
    plt.figure(figsize=(6,4));plt.plot(loss);plt.title(mode+" Combined Loss Curve");plt.xlabel("Epoch");plt.ylabel("Loss");plt.show()

    report.to_csv(mode+"_combined_report.csv")


# FINAL TEST RESULTS
print("\n######## FINAL TEST RESULTS ########")

for mode in ["transductive","inductive"]:

    yt=np.array(results[mode]["true"])
    yp=np.array(results[mode]["pred"])
    prob=np.array(results[mode]["prob"])

    test_report,test_cm=evaluate_block(yt,yp,prob,mode+" Final Test")
    test_report.to_csv(mode+"_final_test_report.csv")


print("PART 4 COMPLETED")

In [ ]:
import pandas as pd
from scipy import stats
from sklearn.metrics import *
from sklearn.preprocessing import label_binarize
def error_analysis(cm):
    return cm.sum(0)-np.diag(cm),cm.sum(1)-np.diag(cm)

def plot_cm(cm,title,norm=False):
    if norm: cm=cm.astype(float)/np.maximum(cm.sum(1,keepdims=True),1)
    sns.heatmap(cm,annot=True,fmt=".2f" if norm else "d",cmap="Blues",xticklabels=class_names,yticklabels=class_names)
    plt.title(title);plt.show()

def plot_history(h,title):
    for a,b in [("accuracy","val_accuracy"),("loss","val_loss")]:
        if a in h and b in h:
            plt.plot(h[a]);plt.plot(h[b]);plt.title(title+" "+a);plt.legend(["Train","Val"]);plt.show()

def evaluate_final(mode):
    yt=np.array(results[mode]["true"])
    yp=np.array(results[mode]["pred"])
    prob=np.array(results[mode]["prob"])
    print("\n"+mode.upper())
    report=pd.DataFrame(classification_report(yt,yp,target_names=class_names,output_dict=True,zero_division=0)).T
    print(report)
    cm=confusion_matrix(yt,yp);print(cm)
    plot_cm(cm,mode+" CM");plot_cm(cm,mode+" NCM",True)
    yb=label_binarize(yt,classes=np.arange(NUM_CLASSES))
    plt.figure(figsize=(6,5))
    aucs=[]
    for i in range(NUM_CLASSES):
        if len(np.unique(yb[:,i]))>1:
            fpr,tpr,_=roc_curve(yb[:,i],prob[:,i]);a=auc(fpr,tpr);aucs.append(a);plt.plot(fpr,tpr,label=f"{class_names[i]}:{a:.3f}")
    fpr,tpr,_=roc_curve(yb.ravel(),prob.ravel())
    micro=auc(fpr,tpr)
    macro=np.mean(aucs) if aucs else 0
    plt.plot(fpr,tpr,"--",label=f"Micro:{micro:.3f}")
    plt.plot([0,1],[0,1],"k--")
    plt.title(f"{mode} ROC Macro:{macro:.3f}")
    plt.legend();plt.show()
    metrics={"Accuracy":accuracy_score(yt,yp),"Precision":precision_score(yt,yp,average="weighted",zero_division=0),"Recall":recall_score(yt,yp,average="weighted",zero_division=0),"F1":f1_score(yt,yp,average="weighted",zero_division=0),"MCC":matthews_corrcoef(yt,yp),"Kappa":cohen_kappa_score(yt,yp),"LogLoss":log_loss(yt,prob),"Macro_AUC":macro,"Micro_AUC":micro}
    print(pd.Series(metrics))
    fp,fn=error_analysis(cm)
    err=pd.DataFrame({"Class":class_names,"Type_I_FP":fp,"Type_II_FN":fn})
    print(err)
    report.to_csv(f"{mode}_report.csv")
    pd.DataFrame([metrics]).to_csv(f"{mode}_metrics.csv",index=False)
    err.to_csv(f"{mode}_errors.csv",index=False)
    return report

transductive_results=evaluate_final("transductive")
inductive_results=evaluate_final("inductive")

def fold_statistics(mode):
    x=np.array(results[mode]["fold_acc"])
    print(mode,"\nScores:",x,"\nMean:",x.mean(),"\nSD:",x.std(),"\n95CI:",stats.t.interval(.95,len(x)-1,loc=x.mean(),scale=stats.sem(x)))
    for i,h in enumerate(results[mode]["histories"]):
        plot_history(h,f"{mode} Fold {i+1}")

fold_statistics("transductive")
fold_statistics("inductive")

for m in ["transductive","inductive"]:
    c=np.array(results[m]["cnn_time"])
    g=np.array(results[m]["gcn_time"])
    print(m,"\nCNN:",c.mean()/60,"\nGCN:",g.mean()/60,"\nTotal:",(c.mean()+g.mean())/60)

transductive_results.to_csv("Hybrid_GCN_kNN_transductive_report.csv")
inductive_results.to_csv("Hybrid_GCN_kNN_inductive_report.csv")

print("PART 4 COMPLETED")

In [ ]:
# ============================================================
# Hybrid-GCN-kNN
# PART 5: COMPUTATIONAL + STATISTICAL ANALYSIS
# ============================================================

import os,time,psutil
import pandas as pd
import numpy as np
from scipy import stats


# MEMORY MONITOR

def ram_usage():
    return psutil.Process(os.getpid()).memory_info().rss/1024**3


def gpu_memory():
    try:
        gpus=tf.config.list_physical_devices("GPU")
        if not gpus:return 0
        return tf.config.experimental.get_memory_info("GPU:0")["current"]/1024**3
    except:return 0


# MODEL PARAMETERS

def model_parameters(model):

    total=model.count_params()
    trainable=np.sum([np.prod(v.shape) for v in model.trainable_weights])
    size_mb=(total*4)/(1024**2)

    return {
        "Parameters":int(total),
        "Trainable":int(trainable),
        "Size_MB":round(size_mb,3)
    }


# CNN COMPLEXITY

cnn_model,_=build_gcnn4()
cnn_metrics=model_parameters(cnn_model)

print("\nCNN MODEL")
for k,v in cnn_metrics.items():print(k,":",v)


# GCN COMPLEXITY

gcn_model=build_gcn(264,100)
gcn_metrics=model_parameters(gcn_model)

print("\nGCN MODEL")
for k,v in gcn_metrics.items():print(k,":",v)



# TRAINING TIME

def training_summary(mode):

    cnn=np.array(results[mode]["cnn_time"])
    gcn=np.array(results[mode]["gcn_time"])

    return {
        "CNN_time_mean_sec":np.mean(cnn),
        "CNN_time_std_sec":np.std(cnn),
        "GCN_time_mean_sec":np.mean(gcn),
        "GCN_time_std_sec":np.std(gcn),
        "Total_mean_sec":np.mean(cnn+gcn)
    }

print("\nTRAINING COST")
for mode in ["transductive","inductive"]:
    print("\n",mode.upper())
    print(training_summary(mode))

# INFERENCE TIME

def inference_speed(model,X,A):

    start=time.time()
    pred=model.predict([X,A],verbose=0)
    total=time.time()-start
    images=len(X)

    return {
        "Inference_seconds":total,
        "Latency_ms":(total/images)*1000,
        "Throughput_img_sec":images/total
    }

# PREDICTION CSV
def save_predictions(mode,image_indices):

    y_true=np.array(results[mode]["true"])
    y_pred=np.array(results[mode]["pred"])
    y_prob=np.array(results[mode]["prob"])

    df=pd.DataFrame()
    
    images=np.concatenate([paths[image_indices] for _ in range(len(y_true)//len(image_indices))])
    df["Image"]=images[:len(y_true)]
    df["True_Label"]=[class_names[i] for i in y_true]
    df["Predicted_Label"]=[class_names[i] for i in y_pred]

    for i,c in enumerate(class_names):
        df[f"Probability_{c}"]=y_prob[:,i]

    df["Correct"]=y_true==y_pred

    filename=f"{mode}_predictions.csv"
    df.to_csv(filename,index=False)

    print("Saved:",filename)

save_predictions("transductive",test_indices)
save_predictions("inductive",test_indices)



# STATISTICAL COMPARISON
def compare_models():
    trans=np.array(results["transductive"]["fold_acc"])
    indu=np.array(results["inductive"]["fold_acc"])

    print("\nMODEL COMPARISON")
    print("Transductive:",trans)
    print("Inductive:",indu)

    t,p_t=stats.ttest_rel(trans,indu)
    w,p_w=stats.wilcoxon(trans,indu)

    print("\nPaired t-test")
    print("t:",t,"p:",p_t)

    print("\nWilcoxon")
    print("W:",w,"p:",p_w)


compare_models()



# FINAL SUMMARY TABLE

summary=[]

for mode in ["transductive","inductive"]:

    acc=np.array(results[mode]["fold_acc"])

    summary.append({
        "Model":mode,
        "Accuracy_mean":np.mean(acc),
        "Accuracy_SD":np.std(acc),
        "95_CI":stats.t.interval(
            0.95,
            len(acc)-1,
            loc=np.mean(acc),
            scale=stats.sem(acc)
        )
    })


summary_df=pd.DataFrame(summary)

print("\nFINAL SUMMARY")
print(summary_df)

summary_df.to_csv("Hybrid_GCN_kNN_final_summary.csv",index=False)

print("\nPART 5 COMPLETED")